# 📊 Telecom Customer Churn Analysis
**Project Summary:**  
This project analyzes a telecom company's customer dataset (7,043 records) to identify key drivers of customer churn. Using exploratory data analysis (EDA), visualizations, and statistical insights, we uncover which customer segments, services, and contract types are most at risk of churning — providing actionable business recommendations to improve customer retention.

---
**Dataset:** Telecom Customer Churn (IBM Sample Dataset)  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Author:** Your Name  
**Date:** 2025


## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visual style
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 100

# Load data
df = pd.read_csv('Customer Churn.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Data Exploration

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data Cleaning

**Issues identified:**
- `TotalCharges` is stored as string (object) — whitespace entries represent new customers (tenure=0) with no charges yet
- `SeniorCitizen` is stored as 0/1 integers — should be mapped to Yes/No for clarity

**Correction:** Replace whitespace TotalCharges with 0 (not 1, not NaN-fill with 1), and convert to float.

In [ ]:
# Fix TotalCharges: convert whitespace to 0 (new customers have no charges yet)
df['TotalCharges'] = df['TotalCharges'].replace(' ', '0')
df['TotalCharges'] = df['TotalCharges'].astype(float)

# Map SeniorCitizen to readable labels
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

# Check for nulls
print('Null values after cleaning:')
print(df.isnull().sum().sum(), 'total nulls')

# Check for duplicates
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Duplicate customer IDs: {df["customerID"].duplicated().sum()}')

df.head()

## 4. Churn Overview

In [ ]:
# Overall churn distribution
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print('Churn Distribution:')
print(pd.DataFrame({'Count': churn_counts, 'Percentage': churn_pct.round(2)}))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
ax = sns.countplot(data=df, x='Churn', ax=axes[0], palette=['#2ecc71', '#e74c3c'])
ax.bar_label(ax.containers[0])
ax.bar_label(ax.containers[1])
axes[0].set_title('Churn Count', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Number of Customers')

# Pie chart
axes[1].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90, explode=(0, 0.05))
axes[1].set_title('Churn Rate (%)', fontsize=14, fontweight='bold')

plt.suptitle('Customer Churn Overview', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n⚠️ Churn Rate: {churn_pct['Yes']:.1f}% — roughly 1 in 4 customers are churning.")

## 5. Demographic Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
demographic_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']
titles = ['Gender', 'Senior Citizen', 'Has Partner', 'Has Dependents']

for ax, col, title in zip(axes.flatten(), demographic_cols, titles):
    chart = sns.countplot(data=df, x=col, hue='Churn', ax=ax,
                          palette={'No': '#2ecc71', 'Yes': '#e74c3c'})
    for container in chart.containers:
        chart.bar_label(container, fontsize=9)
    ax.set_title(f'Churn by {title}', fontsize=12, fontweight='bold')
    ax.set_xlabel(title)
    ax.set_ylabel('Count')
    ax.legend(title='Churn')

plt.suptitle('Churn by Demographics', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Contract & Payment Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Churn by Contract Type
ax1 = sns.countplot(data=df, x='Contract', hue='Churn', ax=axes[0],
                    palette={'No': '#2ecc71', 'Yes': '#e74c3c'})
for c in ax1.containers:
    ax1.bar_label(c)
axes[0].set_title('Churn by Contract Type', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Contract Type')
axes[0].set_ylabel('Number of Customers')

# Churn by Payment Method
ax2 = sns.countplot(data=df, x='PaymentMethod', hue='Churn', ax=axes[1],
                    palette={'No': '#2ecc71', 'Yes': '#e74c3c'})
for c in ax2.containers:
    ax2.bar_label(c)
axes[1].set_title('Churn by Payment Method', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Payment Method')
axes[1].set_ylabel('Number of Customers')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Churn rate by contract
contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).round(1).reset_index()
contract_churn.columns = ['Contract', 'Churn Rate (%)']
print('\nChurn Rate by Contract Type:')
print(contract_churn.to_string(index=False))

## 7. Internet Service & Add-ons Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
service_cols = ['InternetService', 'OnlineSecurity', 'TechSupport',
                'StreamingTV', 'StreamingMovies', 'OnlineBackup']

for ax, col in zip(axes.flatten(), service_cols):
    chart = sns.countplot(data=df, x=col, hue='Churn', ax=ax,
                          palette={'No': '#2ecc71', 'Yes': '#e74c3c'})
    for container in chart.containers:
        chart.bar_label(container, fontsize=8)
    ax.set_title(f'Churn by {col}', fontsize=11, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Churn by Internet Services & Add-ons', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Tenure & Charges Analysis (IMPROVEMENT: Distributions & KDE)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
titles = ['Tenure (Months)', 'Monthly Charges ($)', 'Total Charges ($)']

for ax, col, title in zip(axes, num_cols, titles):
    for churn_val, color in zip(['No', 'Yes'], ['#2ecc71', '#e74c3c']):
        subset = df[df['Churn'] == churn_val][col]
        ax.hist(subset, bins=30, alpha=0.5, color=color, label=f'Churn={churn_val}', density=True)
    ax.set_title(f'Distribution of {title} by Churn', fontsize=12, fontweight='bold')
    ax.set_xlabel(title)
    ax.set_ylabel('Density')
    ax.legend()

plt.tight_layout()
plt.show()

# Summary statistics by churn
print('\nAverage values by Churn status:')
print(df.groupby('Churn')[['tenure', 'MonthlyCharges', 'TotalCharges']].mean().round(2))

## 9. Churn Rate Heatmap by Contract & Internet Service (IMPROVEMENT: New Analysis)

In [ ]:
# Pivot table: churn rate by Contract vs Internet Service
pivot = df.groupby(['Contract', 'InternetService'])['Churn'].apply(
    lambda x: round((x == 'Yes').sum() / len(x) * 100, 1)
).unstack()

plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.5, cbar_kws={'label': 'Churn Rate (%)'})
plt.title('Churn Rate (%) by Contract Type & Internet Service', fontsize=14, fontweight='bold')
plt.xlabel('Internet Service')
plt.ylabel('Contract Type')
plt.tight_layout()
plt.show()

## 10. Correlation Analysis (IMPROVEMENT: New Analysis)

In [ ]:
# Encode churn as binary for correlation
df_corr = df.copy()
df_corr['Churn_Binary'] = (df_corr['Churn'] == 'Yes').astype(int)

# Encode key categorical columns
encode_map = {
    'gender': {'Male': 1, 'Female': 0},
    'SeniorCitizen': {'Yes': 1, 'No': 0},
    'Partner': {'Yes': 1, 'No': 0},
    'Dependents': {'Yes': 1, 'No': 0},
    'PaperlessBilling': {'Yes': 1, 'No': 0},
}
for col, mapping in encode_map.items():
    df_corr[col] = df_corr[col].map(mapping)

num_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'gender',
                'SeniorCitizen', 'Partner', 'Dependents', 'PaperlessBilling', 'Churn_Binary']

corr_matrix = df_corr[num_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Correlation Heatmap (including Churn)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop features correlated with Churn:')
print(corr_matrix['Churn_Binary'].drop('Churn_Binary').sort_values(key=abs, ascending=False))

## 11. Tenure Segmentation (IMPROVEMENT: New Analysis)

In [ ]:
# Create tenure segments
df['TenureGroup'] = pd.cut(df['tenure'],
                            bins=[0, 12, 24, 48, 72],
                            labels=['0-12 months', '13-24 months', '25-48 months', '49-72 months'])

# Churn rate by tenure group
tenure_churn = df.groupby('TenureGroup')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
tenure_churn.columns = ['TenureGroup', 'ChurnRate']

plt.figure(figsize=(9, 5))
bars = plt.bar(tenure_churn['TenureGroup'].astype(str),
               tenure_churn['ChurnRate'],
               color=['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71'])
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             f'{bar.get_height():.1f}%', ha='center', va='bottom', fontweight='bold')
plt.title('Churn Rate by Tenure Segment', fontsize=14, fontweight='bold')
plt.xlabel('Customer Tenure Group')
plt.ylabel('Churn Rate (%)')
plt.tight_layout()
plt.show()

print(tenure_churn.to_string(index=False))

## 12. Key Insights & Business Recommendations

### 🔍 Key Findings:
1. **~26% overall churn rate** — 1 in 4 customers is leaving.
2. **Month-to-month contracts** have dramatically higher churn than 1-year or 2-year contracts.
3. **Fiber optic internet** customers churn more, possibly due to price or service quality expectations.
4. **Customers without OnlineSecurity or TechSupport** are significantly more likely to churn.
5. **Electronic check payment** users show the highest churn among payment methods.
6. **New customers (0-12 months)** churn at the highest rate — early retention is critical.
7. **Senior citizens** churn more than non-senior customers.
8. **Higher MonthlyCharges** correlates with churn; TotalCharges is inversely correlated (loyal customers pay more over time).

### ✅ Business Recommendations:
- Offer **incentives to switch** from month-to-month to annual contracts (discounts, perks).
- Launch a **"first 6 months" onboarding program** to reduce early churn.
- Bundle **OnlineSecurity and TechSupport** into standard plans or offer them at discounted rates.
- Investigate **Fiber optic pricing or service quality** issues — customer satisfaction surveys could help.
- Create **senior citizen loyalty programs** with tailored support.
- Encourage **auto-pay (bank transfer/credit card)** over electronic check with small billing discounts.
